# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.7 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 2880
Session ID: 5d70b57b-bf12-4367-a3b5-c4fe54a25fb2
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
Waiting for session 5d70b57b-bf12-4367-a3b5-c4fe54a25fb2 to get into ready status...
Session 5d70b57b-bf12-4367-a3b5-c4fe54a25fb2 ha

In [2]:
dyf = glueContext.create_dynamic_frame_from_catalog(
    database='lds_raw',
    table_name='consolidado_mensual'
)

df = dyf.toDF()
df.show(10)
df.printSchema()


+-------------+----------+--------+-------------+------------+-------------+-------------+---------------+
|id_suministro|id_medidor|anio_mes|energia_valle|energia_pico|energia_media|energia_total|monto_facturado|
+-------------+----------+--------+-------------+------------+-------------+-------------+---------------+
|      1000000|   2000000| 2021-01|        107.6|       86.89|         67.4|       261.89|         171.23|
|      1000000|   2000000| 2021-02|        86.65|      106.72|        55.74|       249.12|         162.93|
|      1000000|   2000000| 2021-03|        99.17|      112.12|        39.62|       250.91|         164.09|
|      1000000|   2000000| 2021-04|        84.99|       69.22|        45.43|       199.64|         130.76|
|      1000000|   2000000| 2021-05|       137.07|       91.89|        61.18|       290.13|         189.59|
|      1000000|   2000000| 2021-06|       166.79|      115.07|        61.29|       343.15|         224.05|
|      1000000|   2000000| 2021-07|  

In [3]:
total = df.count()
print("Total filas consolidadas:", total)


Total filas consolidadas: 573494


In [4]:
from pyspark.sql import functions as F

print("=== NULOS POR COLUMNA ===")
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()


=== NULOS POR COLUMNA ===
+-------------+----------+--------+-------------+------------+-------------+-------------+---------------+
|id_suministro|id_medidor|anio_mes|energia_valle|energia_pico|energia_media|energia_total|monto_facturado|
+-------------+----------+--------+-------------+------------+-------------+-------------+---------------+
|            0|         0|       0|        10982|       10982|        10982|        10982|          10982|
+-------------+----------+--------+-------------+------------+-------------+-------------+---------------+


In [5]:
string_cols = [c for c, t in df.dtypes if t == "string"]

print("=== BLANCOS POR COLUMNA ===")
df.select([
    F.count(F.when(F.col(c) == "", c)).alias(c)
    for c in string_cols
]).show()


=== BLANCOS POR COLUMNA ===
+--------+
|anio_mes|
+--------+
|       0|
+--------+


In [6]:
df = df.withColumn("anio", F.substring("anio_mes", 1, 4))

df.groupBy("anio").count().orderBy("anio").show()


+----+------+
|anio| count|
+----+------+
|2021|142179|
|2022|142786|
|2023|143610|
|2024|144919|
+----+------+


In [7]:
df.filter(F.col("energia_total") < 0).count()


0


In [8]:
df.select(
    F.min("monto_facturado"),
    F.max("monto_facturado")
).show()


+--------------------+--------------------+
|min(monto_facturado)|max(monto_facturado)|
+--------------------+--------------------+
|               49.22|             1726.32|
+--------------------+--------------------+
